# Regression Metrics - Assignment

Welcome to the assignment for Regression Metrics.

Understanding how to evaluate regression models is fundamental to machine learning. While models can make predictions, measuring the quality of those predictions requires appropriate metrics. Different regression metrics highlight different aspects of model performance, and selecting the right metric depends on your problem domain and business objectives.

In this assignment, you will explore various regression metrics, learn when to use each one, and understand how to diagnose model issues through residual analysis. You'll work with error-based metrics like MSE and MAE, variance-based metrics like R², and learn how to create custom metrics tailored to specific business needs.

Regression metrics are critical in real-world applications where prediction accuracy directly impacts business outcomes. In real estate, prediction errors affect pricing strategies. In demand forecasting, under-prediction leads to stockouts while over-prediction causes excess inventory. Understanding these metrics allows you to align model performance with business requirements.

**What You Will Do in This Assignment**

* Calculate and interpret error-based metrics: MSE, RMSE, MAE.
* Understand R-squared and Adjusted R-squared for explained variance.
* Perform residual analysis to validate model assumptions.
* Compare models using multiple metrics simultaneously.
* Define custom loss functions for business-specific objectives.
* Implement all regression metrics from scratch using NumPy.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the save icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

1. [Introduction to Regression Metrics](#1)
2. [Error-Based Metrics](#2)
   - [Exercise 1: Calculate and Compare Error Metrics](#ex-1)
3. [R-Squared and Adjusted R-Squared](#3)
   - [Exercise 2: Explained Variance Analysis](#ex-2)
4. [Residual Analysis](#4)
   - [Exercise 3: Diagnose Model with Residuals](#ex-3)
5. [Metric Comparison](#5)
   - [Exercise 4: Compare Models with Multiple Metrics](#ex-4)
6. [Custom Metrics](#6)
   - [Exercise 5: Business-Specific Loss Functions](#ex-5)
7. [From-Scratch Implementation](#7)
   - [Exercise 6: Implement All Metrics from Scratch](#ex-6)
8. [Conclusion](#8)

<a name='1'></a>
## 1 - Introduction to Regression Metrics

**Regression metrics** quantify how well predicted continuous values match actual values. Unlike classification metrics that measure correctness of discrete predictions, regression metrics measure the magnitude of prediction errors.

### Core Concepts:

**Residuals**: The difference between actual and predicted values
$$e_i = y_i - \hat{y}_i$$

**Mean Squared Error (MSE)**: Average of squared errors
$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Root Mean Squared Error (RMSE)**: Square root of MSE
$$\text{RMSE} = \sqrt{\text{MSE}} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

**Mean Absolute Error (MAE)**: Average of absolute errors
$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**R-Squared (R²)**: Proportion of variance explained
$$R^2 = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2} = 1 - \frac{SS_{res}}{SS_{tot}}$$

### Key Differences:

**MSE vs MAE**:
- MSE penalizes large errors more heavily (squared term)
- MAE treats all errors equally
- MSE units: squared units of target
- MAE units: same units as target

**RMSE vs MAE**:
- RMSE in same units as target (like MAE)
- RMSE more sensitive to outliers
- MAE more robust to outliers

**R² Interpretation**:
- R² = 1: Perfect predictions
- R² = 0: Model performs like mean baseline
- R² < 0: Model worse than mean baseline

### Metric Selection:

- **Outliers present**: Use MAE (robust)
- **Large errors very costly**: Use MSE or RMSE
- **Need interpretability**: Use MAE or RMSE (same units)
- **Compare models**: Use R² or Adjusted R²
- **Business-specific costs**: Use custom metrics

### Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.datasets import fetch_california_housing, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    mean_absolute_percentage_error, median_absolute_error
)
from scipy import stats

# Set random seed
np.random.seed(42)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("All libraries imported successfully!")

### Helper Functions

In [ ]:
def plot_predictions(y_true, y_pred, title="Predicted vs Actual"):
    """Plot predicted vs actual values with perfect prediction line."""
    plt.figure(figsize=(10, 6))
    plt.scatter(y_true, y_pred, alpha=0.5, edgecolors='k', linewidth=0.5)
    
    # Perfect prediction line
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    plt.xlabel('Actual Values', fontsize=12)
    plt.ylabel('Predicted Values', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_residuals(y_true, y_pred, title="Residual Plot"):
    """Plot residuals vs predicted values."""
    residuals = y_true - y_pred
    
    plt.figure(figsize=(10, 6))
    plt.scatter(y_pred, residuals, alpha=0.5, edgecolors='k', linewidth=0.5)
    plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
    plt.xlabel('Predicted Values', fontsize=12)
    plt.ylabel('Residuals', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_residual_distribution(y_true, y_pred, title="Residual Distribution"):
    """Plot histogram and Q-Q plot of residuals."""
    residuals = y_true - y_pred
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(residuals, bins=50, edgecolor='k', alpha=0.7)
    axes[0].axvline(x=0, color='r', linestyle='--', linewidth=2)
    axes[0].set_xlabel('Residuals', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('Residual Histogram', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Q-Q plot
    stats.probplot(residuals, dist="norm", plot=axes[1])
    axes[1].set_title('Q-Q Plot', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_metric_comparison(models_dict, metrics_dict):
    """Plot comparison of multiple models across multiple metrics."""
    df = pd.DataFrame(metrics_dict).T
    
    fig, axes = plt.subplots(1, len(df.columns), figsize=(5*len(df.columns), 5))
    if len(df.columns) == 1:
        axes = [axes]
    
    for idx, metric in enumerate(df.columns):
        df[metric].plot(kind='bar', ax=axes[idx], color='skyblue', edgecolor='k')
        axes[idx].set_title(metric, fontsize=14, fontweight='bold')
        axes[idx].set_ylabel('Score', fontsize=12)
        axes[idx].set_xlabel('Model', fontsize=12)
        axes[idx].grid(True, alpha=0.3, axis='y')
        axes[idx].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()


print("Helper functions defined!")

### Load Dataset

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing()
X = housing.data
y = housing.target  # Median house value in $100,000s

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dataset loaded successfully!")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nFeatures: {housing.feature_names}")
print(f"Target: Median house value (in $100,000s)")
print(f"\nTarget statistics:")
print(f"  Min:  ${y_train.min():.2f}00k")
print(f"  Max:  ${y_train.max():.2f}00k")
print(f"  Mean: ${y_train.mean():.2f}00k")
print(f"  Std:  ${y_train.std():.2f}00k")

In [ ]:
# Train baseline model
baseline_model = LinearRegression()
baseline_model.fit(X_train_scaled, y_train)

# Get predictions
y_pred_train = baseline_model.predict(X_train_scaled)
y_pred_test = baseline_model.predict(X_test_scaled)

print("Baseline model trained!")
print(f"Training R²: {r2_score(y_train, y_pred_train):.4f}")
print(f"Test R²: {r2_score(y_test, y_pred_test):.4f}")

<a name='2'></a>
## 2 - Error-Based Metrics

**Error-based metrics** measure the deviation between predictions and actual values. These are the most fundamental regression metrics.

### Mean Squared Error (MSE):

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

**Properties:**
- Always non-negative (≥ 0)
- Heavily penalizes large errors (quadratic)
- Units: squared units of target variable
- Differentiable (useful for gradient-based optimization)
- Sensitive to outliers

### Root Mean Squared Error (RMSE):

$$\text{RMSE} = \sqrt{\text{MSE}}$$

**Properties:**
- Same units as target variable (interpretable)
- Still penalizes large errors more than MAE
- Standard deviation of residuals
- Sensitive to outliers

### Mean Absolute Error (MAE):

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**Properties:**
- Same units as target variable
- Linear penalty for errors
- Robust to outliers
- Not differentiable at zero (optimization challenge)

### When to Use Each:

**Use MSE/RMSE when:**
- Large errors are disproportionately costly
- Training neural networks (differentiable loss)
- Normal error distribution expected

**Use MAE when:**
- All errors equally important
- Outliers present in data
- Need robust metric
- Interpretability important

### Relationship:

For any dataset: $\text{MAE} \leq \text{RMSE}$

Large difference indicates presence of outliers or high variance in errors.

<a name='ex-1'></a>
### Exercise 1 - Calculate and Compare Error Metrics

**Your Task:**

Implement the `calculate_error_metrics` function that:
1. Calculates MSE, RMSE, MAE manually
2. Verifies against sklearn implementations
3. Analyzes the relationship between metrics
4. Interprets differences in context of the problem
5. Visualizes predictions and errors

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For manual calculation:**
* Residuals: `residuals = y_true - y_pred`
* MSE: `np.mean(residuals**2)`
* RMSE: `np.sqrt(mse)`
* MAE: `np.mean(np.abs(residuals))`

**For sklearn verification:**
* Use: `mean_squared_error(y_true, y_pred)`
* Use: `mean_squared_error(y_true, y_pred, squared=False)` for RMSE
* Use: `mean_absolute_error(y_true, y_pred)`

**For interpretation:**
* Convert to original units (multiply by $100k)
* Calculate RMSE/MAE ratio to detect outliers
* Compare to target standard deviation

</details>

In [ ]:
# GRADED FUNCTION: calculate_error_metrics
def calculate_error_metrics(y_true, y_pred):
    """
    Calculate error-based regression metrics.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
    
    Returns:
        Dictionary with all error metrics
    """
    ### START CODE HERE ### (≈ 20 lines)
    
    # Calculate residuals
    residuals = None  # y_true - y_pred
    
    # Calculate metrics manually
    mse_manual = None  # Mean of squared residuals
    rmse_manual = None  # Square root of MSE
    mae_manual = None  # Mean of absolute residuals
    
    # Verify with sklearn
    mse_sklearn = None 
    rmse_sklearn = None  
    mae_sklearn = None  
    
    # Calculate additional statistics
    max_error = None  # Maximum absolute error
    median_error = None  # Median absolute error
    rmse_mae_ratio = None  # RMSE/MAE ratio (indicates outliers)
    
    ### END CODE HERE ###
    
    return {
        'mse_manual': mse_manual,
        'rmse_manual': rmse_manual,
        'mae_manual': mae_manual,
        'mse_sklearn': mse_sklearn,
        'rmse_sklearn': rmse_sklearn,
        'mae_sklearn': mae_sklearn,
        'max_error': max_error,
        'median_error': median_error,
        'rmse_mae_ratio': rmse_mae_ratio,
        'residuals': residuals
    }

In [ ]:
# Test your implementation
print("="*60)
print("ERROR METRICS ANALYSIS")
print("="*60)

metrics = calculate_error_metrics(y_test, y_pred_test)

print("\nManual Calculations:")
print(f"  MSE:  {metrics['mse_manual']:.6f}")
print(f"  RMSE: {metrics['rmse_manual']:.6f} ($100k units)")
print(f"  MAE:  {metrics['mae_manual']:.6f} ($100k units)")

print("\nSklearn Verification:")
print(f"  MSE:  {metrics['mse_sklearn']:.6f}")
print(f"  RMSE: {metrics['rmse_sklearn']:.6f}")
print(f"  MAE:  {metrics['mae_sklearn']:.6f}")

print("\nDifferences (should be ~0):")
print(f"  MSE:  {abs(metrics['mse_manual'] - metrics['mse_sklearn']):.10f}")
print(f"  RMSE: {abs(metrics['rmse_manual'] - metrics['rmse_sklearn']):.10f}")
print(f"  MAE:  {abs(metrics['mae_manual'] - metrics['mae_sklearn']):.10f}")

print("\nInterpretation:")
print(f"  Average prediction error: ${metrics['mae_manual']*100:.2f}k")
print(f"  RMSE (std of errors):     ${metrics['rmse_manual']*100:.2f}k")
print(f"  Maximum error:            ${metrics['max_error']*100:.2f}k")
print(f"  Median error:             ${metrics['median_error']*100:.2f}k")
print(f"  RMSE/MAE ratio:           {metrics['rmse_mae_ratio']:.4f}")

if metrics['rmse_mae_ratio'] > 1.5:
    print("  ⚠️  High RMSE/MAE ratio suggests outliers or high error variance")
elif metrics['rmse_mae_ratio'] > 1.2:
    print("  ✓ Moderate RMSE/MAE ratio indicates some error variance")
else:
    print("  ✓ Low RMSE/MAE ratio indicates consistent errors")

# Visualizations
plot_predictions(y_test, y_pred_test, "Predicted vs Actual House Prices")
plot_residuals(y_test, y_pred_test, "Residuals vs Predicted Values")

##### **Expected Output**
```
============================================================
ERROR METRICS ANALYSIS
============================================================

Manual Calculations:
  MSE:  0.XXXXXX
  RMSE: 0.XXXXXX ($100k units)
  MAE:  0.XXXXXX ($100k units)

Sklearn Verification:
  MSE:  0.XXXXXX
  RMSE: 0.XXXXXX
  MAE:  0.XXXXXX

Differences (should be ~0):
  MSE:  0.0000000000
  RMSE: 0.0000000000
  MAE:  0.0000000000

Interpretation:
  Average prediction error: $XXk
  RMSE (std of errors):     $XXk
  Maximum error:            $XXXk
  Median error:             $XXk
  RMSE/MAE ratio:           1.XXXX
```
Your manual calculations should match sklearn exactly. RMSE should be greater than MAE.

In [ ]:
unittests.exercise_1(calculate_error_metrics)

<a name='3'></a>
## 3 - R-Squared and Adjusted R-Squared

**R-squared (R²)** measures the proportion of variance in the target variable that is explained by the model.

### R-Squared (Coefficient of Determination):

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}{\sum_{i=1}^{n} (y_i - \bar{y})^2}$$

where:
- $SS_{res}$ = Residual sum of squares (model error)
- $SS_{tot}$ = Total sum of squares (variance in data)
- $\bar{y}$ = Mean of actual values

**Interpretation:**
- R² = 1.0: Perfect predictions (all variance explained)
- R² = 0.0: Model as good as predicting mean
- R² < 0.0: Model worse than mean (poor fit)
- R² = 0.7: Model explains 70% of variance

### Adjusted R-Squared:

$$R^2_{adj} = 1 - \frac{(1-R^2)(n-1)}{n-p-1}$$

where:
- $n$ = number of samples
- $p$ = number of features

**Purpose:**
- Penalizes adding features that don't improve model
- Accounts for model complexity
- Can decrease when adding irrelevant features
- Better for comparing models with different numbers of features

### Limitations of R²:

1. **Always increases with more features** (even if random)
2. **Not comparable across different datasets**
3. **Doesn't indicate if model is appropriate**
4. **Can be misleading with non-linear relationships**
5. **Sensitive to outliers**

### Good R² Values:

Context-dependent:
- **Physics/Engineering**: R² > 0.9 expected
- **Economics/Social Sciences**: R² > 0.5 often good
- **Behavioral/Biological**: R² > 0.3 can be acceptable

**Always check:**
- Residual plots (more important than R²)
- Training vs test R² (detect overfitting)
- Domain knowledge and expectations

<a name='ex-2'></a>
### Exercise 2 - Explained Variance Analysis

**Your Task:**

Implement the `calculate_r_squared` function that:
1. Calculates R² manually from SS_res and SS_tot
2. Calculates Adjusted R² accounting for features
3. Verifies against sklearn
4. Compares training vs test R² (overfitting check)
5. Tests effect of adding features on Adjusted R²

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For R² calculation:**
* SS_res: `np.sum((y_true - y_pred)**2)`
* SS_tot: `np.sum((y_true - y_true.mean())**2)`
* R²: `1 - (ss_res / ss_tot)`

**For Adjusted R²:**
* Formula: `1 - ((1 - r2) * (n - 1)) / (n - p - 1)`
* n = number of samples
* p = number of features

**For sklearn verification:**
* Use: `r2_score(y_true, y_pred)`

</details>

In [ ]:
# GRADED FUNCTION: calculate_r_squared
def calculate_r_squared(y_true, y_pred, n_features):
    """
    Calculate R² and Adjusted R².
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
        n_features: Number of features used
    
    Returns:
        Dictionary with R² metrics
    """
    ### START CODE HERE ### (≈ 15 lines)
    
    # Calculate sum of squares
    ss_res = None  # Residual sum of squares
    ss_tot = None  # Total sum of squares
    
    # Calculate R² manually
    r2_manual = None  
    
    # Calculate Adjusted R²
    n = None  # Number of samples
    p = None
    r2_adjusted = None  # Apply adjustment formula
    
    # Verify with sklearn
    r2_sklearn = None  
    
    # Calculate explained variance
    variance_explained = None  
    
    ### END CODE HERE ###
    
    return {
        'r2_manual': r2_manual,
        'r2_adjusted': r2_adjusted,
        'r2_sklearn': r2_sklearn,
        'variance_explained': variance_explained,
        'ss_res': ss_res,
        'ss_tot': ss_tot
    }

In [ ]:
# Test your implementation
print("="*60)
print("R-SQUARED ANALYSIS")
print("="*60)

# Training set
r2_train = calculate_r_squared(y_train, y_pred_train, X_train.shape[1])
print("\nTraining Set:")
print(f"  R² (manual):     {r2_train['r2_manual']:.6f}")
print(f"  R² (sklearn):    {r2_train['r2_sklearn']:.6f}")
print(f"  Adjusted R²:     {r2_train['r2_adjusted']:.6f}")
print(f"  Variance Explained: {r2_train['variance_explained']:.2f}%")

# Test set
r2_test = calculate_r_squared(y_test, y_pred_test, X_test.shape[1])
print("\nTest Set:")
print(f"  R² (manual):     {r2_test['r2_manual']:.6f}")
print(f"  R² (sklearn):    {r2_test['r2_sklearn']:.6f}")
print(f"  Adjusted R²:     {r2_test['r2_adjusted']:.6f}")
print(f"  Variance Explained: {r2_test['variance_explained']:.2f}%")

print("\nOverfitting Check:")
r2_diff = r2_train['r2_manual'] - r2_test['r2_manual']
print(f"  Train R² - Test R²: {r2_diff:.6f}")
if r2_diff > 0.1:
    print("  ⚠️  Significant difference suggests overfitting")
elif r2_diff > 0.05:
    print("  ⚠️  Moderate difference, monitor for overfitting")
else:
    print("  ✓ Small difference, good generalization")

print("\nDifference (Manual vs Sklearn):")
print(f"  {abs(r2_test['r2_manual'] - r2_test['r2_sklearn']):.10f}")

In [ ]:
# Demonstrate effect of adding features
print("\n" + "="*60)
print("ADJUSTED R² WITH DIFFERENT FEATURE COUNTS")
print("="*60)

results = []

# Test with 2, 4, 6, 8 features
for n_feat in [2, 4, 6, 8]:
    # Train model with subset of features
    model = LinearRegression()
    model.fit(X_train_scaled[:, :n_feat], y_train)
    y_pred = model.predict(X_test_scaled[:, :n_feat])
    
    metrics = calculate_r_squared(y_test, y_pred, n_feat)
    results.append({
        'n_features': n_feat,
        'R²': metrics['r2_manual'],
        'Adjusted R²': metrics['r2_adjusted']
    })

df_features = pd.DataFrame(results)
print("\n", df_features.to_string(index=False))

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = df_features['n_features']
ax.plot(x, df_features['R²'], marker='o', linewidth=2, label='R²', markersize=8)
ax.plot(x, df_features['Adjusted R²'], marker='s', linewidth=2, label='Adjusted R²', markersize=8)
ax.set_xlabel('Number of Features', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('R² vs Adjusted R² with Different Feature Counts', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservation:")
print("  R² increases or stays constant as features are added.")
print("  Adjusted R² can decrease if new features don't improve model enough.")

##### **Expected Output**
```
============================================================
R-SQUARED ANALYSIS
============================================================

Training Set:
  R² (manual):     0.XXXXXX
  R² (sklearn):    0.XXXXXX
  Adjusted R²:     0.XXXXXX
  Variance Explained: XX.XX%

Test Set:
  R² (manual):     0.XXXXXX
  R² (sklearn):    0.XXXXXX
  Adjusted R²:     0.XXXXXX
  Variance Explained: XX.XX%
```
Your R² should be between 0 and 1. Adjusted R² should be slightly lower than R². Manual and sklearn should match exactly.

In [ ]:
unittests.exercise_2(calculate_r_squared)

<a name='4'></a>
## 4 - Residual Analysis

**Residual analysis** is crucial for validating regression model assumptions and diagnosing problems.

### What are Residuals?

$$e_i = y_i - \hat{y}_i$$

Residuals are the errors between actual and predicted values. Analyzing their patterns reveals model issues.

### Key Assumptions to Check:

**1. Linearity:**
- Relationship between features and target is linear
- **Check**: Residuals vs Predicted plot should show random scatter
- **Violation**: Curved pattern indicates non-linear relationship

**2. Homoscedasticity (Constant Variance):**
- Error variance is constant across all prediction levels
- **Check**: Residuals vs Predicted plot should have constant spread
- **Violation**: Funnel shape indicates heteroscedasticity

**3. Normality of Residuals:**
- Residuals follow normal distribution
- **Check**: Q-Q plot, histogram
- **Violation**: Skewed histogram, points off diagonal in Q-Q plot

**4. Independence:**
- Residuals are independent (no autocorrelation)
- **Check**: Residuals vs order/time plot
- **Violation**: Patterns over time/order

### Diagnostic Plots:

**Residuals vs Predicted:**
- Random scatter → Good
- Curved pattern → Non-linearity
- Funnel shape → Heteroscedasticity
- Points far from zero → Outliers

**Q-Q Plot:**
- Points on diagonal → Normal distribution
- Points off diagonal → Non-normality
- Heavy tails → Outliers present

**Histogram of Residuals:**
- Bell-shaped, centered at zero → Good
- Skewed → Non-normality
- Multiple peaks → Mixture of populations

### Common Issues:

**Outliers:**
- Residuals with large absolute values
- Can distort model fit
- **Solution**: Investigate, remove if errors, or use robust methods

**High Leverage Points:**
- Extreme feature values
- Can influence model disproportionately
- **Check**: Cook's distance, leverage plots

**Non-linearity:**
- **Solution**: Transform features, add polynomial terms, use non-linear model

**Heteroscedasticity:**
- **Solution**: Transform target (log), weighted regression, robust standard errors

<a name='ex-3'></a>
### Exercise 3 - Diagnose Model with Residuals

**Your Task:**

Implement the `analyze_residuals` function that:
1. Calculates residuals and statistics
2. Tests for normality (Shapiro-Wilk test)
3. Identifies outliers (beyond 3 standard deviations)
4. Checks for patterns in residuals
5. Creates diagnostic plots

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For residual calculation:**
* Residuals: `y_true - y_pred`
* Statistics: `residuals.mean()`, `residuals.std()`

**For normality test:**
* Use: `stats.shapiro(residuals)`
* Returns: statistic, p-value
* p-value > 0.05 suggests normality

**For outliers:**
* Threshold: `threshold = 3 * residuals.std()`
* Outliers: `np.abs(residuals) > threshold`
* Count: `outliers.sum()`

**For patterns:**
* Check correlation: `np.corrcoef(y_pred, residuals)[0, 1]`
* Should be near zero

</details>

In [ ]:
# GRADED FUNCTION: analyze_residuals
def analyze_residuals(y_true, y_pred):
    """
    Perform comprehensive residual analysis.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
    
    Returns:
        Dictionary with residual analysis results
    """
    ### START CODE HERE ### (≈ 25 lines)
    
    # Calculate residuals
    residuals = None  
    
    # Basic statistics
    mean_residual = None  # Should be near zero
    std_residual = None
    
    # Normality test (Shapiro-Wilk)
    # Sample if too many points (Shapiro fails on large datasets)
    sample_size = min(5000, len(residuals))
    sample_indices = np.random.choice(len(residuals), sample_size, replace=False)
    shapiro_stat, shapiro_pvalue = None, None  # stats.shapiro(residuals[sample_indices])
    is_normal = None  # shapiro_pvalue > 0.05
    
    # Outlier detection (3 sigma rule)
    threshold = None  
    outliers = None  
    outlier_percentage = None  
    
    # Check for patterns (correlation between predictions and residuals)
    correlation = None  
    
    # Check homoscedasticity (Breusch-Pagan test approximation)
    # Simple check: split predictions into quartiles, compare residual variances
    quartiles = np.percentile(y_pred, [25, 50, 75])
    q1_var = np.var(residuals[y_pred <= quartiles[0]])
    q4_var = np.var(residuals[y_pred >= quartiles[2]])
    variance_ratio = None  
    is_homoscedastic = None  
    
    ### END CODE HERE ###
    
    return {
        'residuals': residuals,
        'mean': mean_residual,
        'std': std_residual,
        'shapiro_stat': shapiro_stat,
        'shapiro_pvalue': shapiro_pvalue,
        'is_normal': is_normal,
        'n_outliers': n_outliers,
        'outlier_percentage': outlier_percentage,
        'outlier_indices': np.where(outliers)[0],
        'correlation': correlation,
        'variance_ratio': variance_ratio,
        'is_homoscedastic': is_homoscedastic
    }

In [ ]:
# Test your implementation
print("="*60)
print("RESIDUAL ANALYSIS")
print("="*60)

residual_analysis = analyze_residuals(y_test, y_pred_test)

print("\nResidual Statistics:")
print(f"  Mean:     {residual_analysis['mean']:.6f} (should be ≈ 0)")
print(f"  Std Dev:  {residual_analysis['std']:.6f}")

print("\nNormality Test (Shapiro-Wilk):")
print(f"  Statistic: {residual_analysis['shapiro_stat']:.6f}")
print(f"  P-value:   {residual_analysis['shapiro_pvalue']:.6f}")
if residual_analysis['is_normal']:
    print("  ✓ Residuals appear normally distributed (p > 0.05)")
else:
    print("  ⚠️  Residuals may not be normally distributed (p ≤ 0.05)")

print("\nOutlier Detection (3-sigma rule):")
print(f"  Number of outliers: {residual_analysis['n_outliers']}")
print(f"  Percentage:         {residual_analysis['outlier_percentage']:.2f}%")
if residual_analysis['outlier_percentage'] > 5:
    print("  ⚠️  High percentage of outliers detected")
else:
    print("  ✓ Outlier percentage is acceptable")

print("\nLinearity Check:")
print(f"  Correlation (predictions vs residuals): {residual_analysis['correlation']:.6f}")
if abs(residual_analysis['correlation']) < 0.1:
    print("  ✓ Low correlation suggests linearity assumption holds")
else:
    print("  ⚠️  Correlation suggests potential non-linearity")

print("\nHomoscedasticity Check:")
print(f"  Variance ratio (Q4/Q1): {residual_analysis['variance_ratio']:.2f}")
if residual_analysis['is_homoscedastic']:
    print("  ✓ Variance appears constant (homoscedastic)")
else:
    print("  ⚠️  Variance may not be constant (heteroscedastic)")

# Create diagnostic plots
print("\nGenerating diagnostic plots...\n")
plot_residuals(y_test, y_pred_test, "Residual Plot - Check for Patterns")
plot_residual_distribution(y_test, y_pred_test, "Residual Distribution Analysis")

##### **Expected Output**
```
============================================================
RESIDUAL ANALYSIS
============================================================

Residual Statistics:
  Mean:     0.XXXXXX (should be ≈ 0)
  Std Dev:  0.XXXXXX

Normality Test (Shapiro-Wilk):
  Statistic: 0.XXXXXX
  P-value:   0.XXXXXX
  ...

Outlier Detection (3-sigma rule):
  Number of outliers: XX
  Percentage:         X.XX%
  ...
```
Mean residual should be very close to zero. Check plots for patterns - random scatter is good, patterns indicate issues.

In [ ]:
unittests.exercise_3(analyze_residuals)

<a name='5'></a>
## 5 - Metric Comparison

**Different metrics capture different aspects of model performance.** Comparing multiple models requires evaluating them on multiple metrics simultaneously.

### Why Multiple Metrics?

**Single metric limitations:**
- MSE: Heavily influenced by outliers
- MAE: May miss large systematic errors
- R²: Can be misleading without residual analysis
- RMSE: Units may not be intuitive

**Comprehensive evaluation needs:**
- Error magnitude (MAE, RMSE)
- Outlier sensitivity (MAE vs RMSE difference)
- Variance explained (R², Adjusted R²)
- Residual patterns (diagnostic plots)

### Model Comparison Framework:

**1. Train multiple models:**
- Linear Regression (baseline)
- Ridge Regression (L2 regularization)
- Lasso Regression (L1 regularization)
- Polynomial features

**2. Calculate all metrics:**
- MSE, RMSE, MAE
- R², Adjusted R²
- Max error, median error

**3. Analyze tradeoffs:**
- Training vs test performance (overfitting)
- Metric discrepancies (outlier presence)
- Complexity vs performance

**4. Select best model:**
- Based on business priorities
- Consider interpretability
- Validate on holdout set

### Metric Discrepancy Analysis:

**RMSE >> MAE:**
- Indicates presence of large errors/outliers
- Model makes occasional big mistakes
- Consider: Robust methods, outlier removal

**RMSE ≈ MAE:**
- Errors are relatively uniform
- Few outliers present
- Model behavior is consistent

**Train R² >> Test R²:**
- Overfitting detected
- Model too complex for data
- Consider: Regularization, feature selection

**Low R² but low MAE:**
- Model predictions accurate on average
- But high variance in target not explained
- May need more features or different model

<a name='ex-4'></a>
### Exercise 4 - Compare Models with Multiple Metrics

**Your Task:**

Implement the `compare_models` function that:
1. Trains multiple regression models
2. Calculates all metrics for each model
3. Creates comparison table and visualizations
4. Analyzes metric discrepancies
5. Recommends best model based on criteria

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For training models:**
* Linear: `LinearRegression().fit(X_train, y_train)`
* Ridge: `Ridge(alpha=1.0).fit(X_train, y_train)`
* Lasso: `Lasso(alpha=0.1).fit(X_train, y_train)`

**For each model, calculate:**
* Train and test predictions
* All error metrics (MSE, RMSE, MAE)
* R² on train and test
* Store in dictionary

**For comparison:**
* Create DataFrame from results
* Find best model per metric: `df[metric].idxmin()` or `idxmax()`

</details>

In [ ]:
# GRADED FUNCTION: compare_models
def compare_models(X_train, X_test, y_train, y_test, models_dict):
    """
    Compare multiple models using various metrics.
    
    Args:
        X_train: Training features
        X_test: Test features
        y_train: Training target
        y_test: Test target
        models_dict: Dictionary of {name: model_instance}
    
    Returns:
        Dictionary with comparison results
    """
    ### START CODE HERE ### (≈ 35 lines)
    
    results = {}
    
    for name, model in models_dict.items():
        # Train model
        model.fit(X_train, y_train)
        
        # Get predictions
        y_pred_train = None  
        y_pred_test = None  
        
        # Calculate all metrics
        train_mse = None  
        test_mse = None
        train_rmse = None  
        test_rmse = None
        train_mae = None  
        test_mae = None
        train_r2 = None 
        test_r2 = None
        
        # Calculate adjusted R²
        n = len(y_test)
        p = X_test.shape[1]
        test_r2_adj = None  
        
        # Calculate RMSE/MAE ratio (outlier indicator)
        ratio = None  
        
        # Store results
        results[name] = {
            'Train MSE': train_mse,
            'Test MSE': test_mse,
            'Train RMSE': train_rmse,
            'Test RMSE': test_rmse,
            'Train MAE': train_mae,
            'Test MAE': test_mae,
            'Train R²': train_r2,
            'Test R²': test_r2,
            'Adj R²': test_r2_adj,
            'RMSE/MAE': ratio
        }
    
    ### END CODE HERE ###
    
    return results

In [ ]:
# Test your implementation
print("="*60)
print("MODEL COMPARISON")
print("="*60)

# Define models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Ridge (α=10)': Ridge(alpha=10),
    'Lasso (α=0.1)': Lasso(alpha=0.1, max_iter=10000)
}

# Compare models
comparison = compare_models(X_train_scaled, X_test_scaled, y_train, y_test, models)

# Create comparison DataFrame
df_comparison = pd.DataFrame(comparison).T

print("\nAll Metrics:")
print(df_comparison.to_string())

# Identify best model for each metric
print("\n" + "="*60)
print("BEST MODEL PER METRIC")
print("="*60)

# Lower is better for error metrics
print(f"\nLowest Test MSE:  {df_comparison['Test MSE'].idxmin()} ({df_comparison['Test MSE'].min():.6f})")
print(f"Lowest Test RMSE: {df_comparison['Test RMSE'].idxmin()} ({df_comparison['Test RMSE'].min():.6f})")
print(f"Lowest Test MAE:  {df_comparison['Test MAE'].idxmin()} ({df_comparison['Test MAE'].min():.6f})")

# Higher is better for R²
print(f"\nHighest Test R²:  {df_comparison['Test R²'].idxmax()} ({df_comparison['Test R²'].max():.6f})")
print(f"Highest Adj R²:   {df_comparison['Adj R²'].idxmax()} ({df_comparison['Adj R²'].max():.6f})")

# Overfitting analysis
print("\n" + "="*60)
print("OVERFITTING ANALYSIS")
print("="*60)
df_comparison['R² Gap'] = df_comparison['Train R²'] - df_comparison['Test R²']
print("\n", df_comparison[['Train R²', 'Test R²', 'R² Gap']].to_string())

# Visualize comparison
print("\nGenerating comparison plots...\n")
plot_metric_comparison(
    models,
    {name: {'Test RMSE': vals['Test RMSE'], 
            'Test MAE': vals['Test MAE'], 
            'Test R²': vals['Test R²']} 
     for name, vals in comparison.items()}
)

##### **Expected Output**
```
============================================================
MODEL COMPARISON
============================================================

All Metrics:
                     Train MSE  Test MSE  Train RMSE  ...
Linear Regression      0.XXXX    0.XXXX     0.XXXX   ...
Ridge (α=1.0)          0.XXXX    0.XXXX     0.XXXX   ...
...

============================================================
BEST MODEL PER METRIC
============================================================

Lowest Test MSE:  ...
```
You should see that regularized models (Ridge/Lasso) may perform better on test set, indicating reduced overfitting.

In [ ]:
unittests.exercise_4(compare_models)

<a name='6'></a>
## 6 - Custom Metrics

**Business-specific problems often require custom loss functions** that align with actual costs and priorities.

### Why Custom Metrics?

Standard metrics (MSE, MAE, R²) may not reflect:
- **Asymmetric costs**: Over-prediction vs under-prediction
- **Non-linear penalties**: Certain error ranges more critical
- **Business constraints**: Minimum accuracy requirements
- **Domain-specific needs**: Industry standards

### Common Custom Metrics:

**1. Asymmetric Loss:**
$$L_{asymm} = \frac{1}{n} \sum_{i=1}^{n} \begin{cases} 
\alpha \cdot (y_i - \hat{y}_i)^2 & \text{if } y_i > \hat{y}_i \\
\beta \cdot (y_i - \hat{y}_i)^2 & \text{if } y_i \leq \hat{y}_i
\end{cases}$$

**Use case**: Demand forecasting
- Under-prediction (stockout): High cost $\alpha$
- Over-prediction (excess inventory): Lower cost $\beta$

**2. Quantile Loss (Pinball Loss):**
$$L_q(y, \hat{y}) = \begin{cases}
q \cdot (y - \hat{y}) & \text{if } y \geq \hat{y} \\
(q-1) \cdot (y - \hat{y}) & \text{if } y < \hat{y}
\end{cases}$$

**Use case**: Quantile regression
- $q = 0.5$: Median (robust to outliers)
- $q = 0.9$: 90th percentile prediction

**3. Percentage-Based Loss:**
$$L_{pct} = \frac{1}{n} \sum_{i=1}^{n} \left|\frac{y_i - \hat{y}_i}{y_i}\right| \times 100$$

**Use case**: When relative error matters
- Same absolute error has different impact at different scales
- $1k error on $10k vs $100k house

**4. Threshold-Based Accuracy:**
$$\text{Acc}_{\epsilon} = \frac{1}{n} \sum_{i=1}^{n} \mathbb{1}\{|y_i - \hat{y}_i| \leq \epsilon\}$$

**Use case**: Binary success/failure
- Predictions within tolerance are "good"
- Outside tolerance are "bad"

### Designing Custom Metrics:

**Steps:**
1. **Identify business costs**: What are actual costs of errors?
2. **Quantify asymmetry**: Are over/under predictions equally bad?
3. **Define thresholds**: What error levels are acceptable?
4. **Implement function**: Code the custom loss
5. **Validate**: Test on known cases
6. **Optimize**: Train models to minimize custom loss

### Example Scenarios:

**Real Estate:**
- Over-pricing: House sits unsold (carrying costs)
- Under-pricing: Money left on table
- Custom: Asymmetric loss with domain knowledge

**Energy Demand:**
- Over-prediction: Excess generation cost
- Under-prediction: Blackouts (catastrophic)
- Custom: Heavily penalize under-prediction

**Medical Dosing:**
- Under-dose: Ineffective treatment
- Over-dose: Toxic side effects
- Custom: Safety margins with non-linear penalty

<a name='ex-5'></a>
### Exercise 5 - Business-Specific Loss Functions

**Your Task:**

Implement custom loss functions for different scenarios:
1. **Asymmetric loss** for inventory management
2. **Quantile loss** for risk-aware predictions
3. **Percentage loss** for relative accuracy
4. **Threshold accuracy** for acceptable error ranges
5. Compare standard vs custom metrics

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For asymmetric loss:**
* Under-prediction: `(y_true > y_pred)`
* Over-prediction: `(y_true <= y_pred)`
* Cost: `cost_under * error**2` for under-predictions
* Cost: `cost_over * error**2` for over-predictions

**For quantile loss:**
* Error: `y_true - y_pred`
* Over-prediction: `quantile * error`
* Under-prediction: `(quantile - 1) * error`
* Use np.where() for conditional calculation

**For percentage loss:**
* MAPE: `np.mean(np.abs((y_true - y_pred) / y_true)) * 100`
* Handle division by zero

**For threshold accuracy:**
* Within threshold: `np.abs(y_true - y_pred) <= threshold`
* Accuracy: `np.mean(within_threshold) * 100`

</details>

#### 5a. `asymmetric_loss` - Asymmetric Loss Function

Implement an asymmetric loss that assigns **different penalty weights** for over- and under-prediction. This is useful in scenarios like inventory management where a stockout (under-predicting demand) costs more than overstocking.

$$L = \frac{1}{n} \sum_{i=1}^{n} \begin{cases} c_{under} \cdot (y_i - \hat{y}_i)^2 & \text{if } y_i > \hat{y}_i \text{ (under-prediction)} \\ c_{over} \cdot (y_i - \hat{y}_i)^2 & \text{if } y_i \leq \hat{y}_i \text{ (over-prediction)} \end{cases}$$

In [ ]:
# GRADED FUNCTION: asymmetric_loss
def asymmetric_loss(y_true, y_pred, cost_under=2.0, cost_over=1.0):
    """
    Calculate asymmetric loss with different costs for over/under prediction.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
        cost_under: Cost multiplier for under-prediction
        cost_over: Cost multiplier for over-prediction
    
    Returns:
        Asymmetric loss value
    """
    ### START CODE HERE ### (≈ 10 lines)
    
    errors = None  
    squared_errors = None  
    
    # Identify under-predictions and over-predictions
    under_pred = None  
    over_pred = None  

    # Calculate weighted loss
    loss = None  
    
    ### END CODE HERE ###
    
    return loss

In [ ]:
unittests.exercise_5a(asymmetric_loss)

#### 5b. `quantile_loss` - Quantile (Pinball) Loss

Implement the **pinball loss** for quantile regression. Unlike MSE, it lets you target a specific quantile of the conditional distribution. Setting `quantile=0.9` penalises under-prediction more heavily, producing a 90th-percentile forecast.

$$L_q(y, \hat{y}) = \frac{1}{n} \sum_{i=1}^{n} \begin{cases} q \cdot (y_i - \hat{y}_i) & \text{if } y_i \geq \hat{y}_i \\ (q - 1) \cdot (y_i - \hat{y}_i) & \text{if } y_i < \hat{y}_i \end{cases}$$

In [ ]:
# GRADED FUNCTION: quantile_loss
def quantile_loss(y_true, y_pred, quantile=0.5):
    """
    Calculate quantile loss (pinball loss).
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
        quantile: Quantile level (0-1)
    
    Returns:
        Quantile loss value
    """
    ### START CODE HERE ### (≈ 8 lines)
    
    errors = None  
    
    # Apply quantile penalty
    loss = None  
    
    ### END CODE HERE ###
    
    return np.mean(loss)

In [ ]:
unittests.exercise_5b(quantile_loss)

#### 5c. `percentage_error` - Mean Absolute Percentage Error (MAPE)

Implement MAPE, which expresses the prediction error as a **percentage of the actual value**. This makes it scale-independent and easy to interpret across different datasets. Be careful to handle cases where `y_true` contains zeros.

$$MAPE = \frac{1}{n} \sum_{i=1}^{n} \left|\frac{y_i - \hat{y}_i}{y_i}\right| \times 100$$

In [ ]:
# GRADED FUNCTION: percentage_error
def percentage_error(y_true, y_pred):
    """
    Calculate Mean Absolute Percentage Error (MAPE).
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
    
    Returns:
        MAPE as percentage
    """
    ### START CODE HERE ### (≈ 8 lines)
    
    # Remove zeros to avoid division by zero
    mask = None  
    y_true_filtered = None  
    y_pred_filtered = None  
    
    # Calculate MAPE
    mape = None  
    ### END CODE HERE ###
    
    return mape

In [ ]:
unittests.exercise_5c(percentage_error)

#### 5d. `threshold_accuracy` - Threshold Accuracy

Implement the **percentage of predictions within an acceptable error tolerance**. This is often more intuitive for stakeholders than MSE — e.g., *"88% of house price predictions are within \$50k of the actual price"*.

$$Acc_\tau = \frac{1}{n} \sum_{i=1}^{n} \mathbf{1}\!\left[|y_i - \hat{y}_i| \leq \tau\right] \times 100$$

In [ ]:
# GRADED FUNCTION: threshold_accuracy
def threshold_accuracy(y_true, y_pred, threshold=0.5):
    """
    Calculate percentage of predictions within threshold.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
        threshold: Acceptable error threshold
    
    Returns:
        Accuracy percentage
    """
    ### START CODE HERE ### (≈ 6 lines)
    
    errors = None  
    within_threshold = None  
    accuracy = None  
    
    ### END CODE HERE ###
    
    return accuracy

In [ ]:
unittests.exercise_5d(threshold_accuracy)

In [ ]:
# Test your implementation
print("="*60)
print("CUSTOM METRICS ANALYSIS")
print("="*60)

print("\n1. ASYMMETRIC LOSS (Inventory Management)")
print("   Under-prediction (stockout) costs 2x more than over-prediction")
asymm_loss = asymmetric_loss(y_test, y_pred_test, cost_under=2.0, cost_over=1.0)
print(f"   Asymmetric Loss: {asymm_loss:.6f}")

# Compare with symmetric MSE
mse = mean_squared_error(y_test, y_pred_test)
print(f"   Standard MSE:    {mse:.6f}")
print(f"   Difference:      {asymm_loss - mse:.6f}")

print("\n2. QUANTILE LOSS (Risk-Aware Predictions)")
for q in [0.1, 0.5, 0.9]:
    q_loss = quantile_loss(y_test, y_pred_test, quantile=q)
    print(f"   {int(q*100)}th percentile loss: {q_loss:.6f}")
print("   Lower quantiles penalize over-prediction more.")
print("   Higher quantiles penalize under-prediction more.")

print("\n3. PERCENTAGE ERROR (Relative Accuracy)")
mape = percentage_error(y_test, y_pred_test)
print(f"   MAPE: {mape:.2f}%")
print(f"   On average, predictions are off by {mape:.2f}% of actual value.")

print("\n4. THRESHOLD ACCURACY (Acceptable Error Range)")
for thresh in [0.25, 0.5, 1.0]:
    acc = threshold_accuracy(y_test, y_pred_test, threshold=thresh)
    print(f"   Within ±${thresh*100}k: {acc:.2f}% of predictions")

# Compare models using custom metric
print("\n" + "="*60)
print("COMPARE MODELS WITH CUSTOM METRIC")
print("="*60)

models_custom = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1, max_iter=10000)
}

results_custom = {}
for name, model in models_custom.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results_custom[name] = {
        'MSE': mean_squared_error(y_test, y_pred),
        'MAE': mean_absolute_error(y_test, y_pred),
        'Asymmetric': asymmetric_loss(y_test, y_pred, cost_under=2.0, cost_over=1.0),
        'MAPE': percentage_error(y_test, y_pred),
        'Acc@0.5': threshold_accuracy(y_test, y_pred, threshold=0.5)
    }

df_custom = pd.DataFrame(results_custom).T
print("\n", df_custom.to_string())

print("\nBest model by custom metrics:")
print(f"  Lowest Asymmetric Loss: {df_custom['Asymmetric'].idxmin()}")
print(f"  Lowest MAPE:            {df_custom['MAPE'].idxmin()}")
print(f"  Highest Acc@0.5:        {df_custom['Acc@0.5'].idxmax()}")

##### **Expected Output**
```
============================================================
CUSTOM METRICS ANALYSIS
============================================================

1. ASYMMETRIC LOSS (Inventory Management)
   Under-prediction (stockout) costs 2x more than over-prediction
   Asymmetric Loss: 0.XXXXXX
   Standard MSE:    0.XXXXXX
   Difference:      0.XXXXXX

2. QUANTILE LOSS (Risk-Aware Predictions)
   10th percentile loss: 0.XXXXXX
   50th percentile loss: 0.XXXXXX
   90th percentile loss: 0.XXXXXX
   ...
```
Custom metrics provide different perspectives on model performance aligned with business objectives.

<a name='7'></a>
## 7 - From-Scratch Implementation: All Regression Metrics

Implementing all regression metrics from scratch deepens understanding of their mathematical foundations and relationships.

### Metrics to Implement:

**1. MSE (Mean Squared Error)**
```python
MSE = (1/n) * Σ(y_true - y_pred)²
```

**2. RMSE (Root Mean Squared Error)**
```python
RMSE = √MSE
```

**3. MAE (Mean Absolute Error)**
```python
MAE = (1/n) * Σ|y_true - y_pred|
```

**4. R² (Coefficient of Determination)**
```python
R² = 1 - (SS_res / SS_tot)
SS_res = Σ(y_true - y_pred)²
SS_tot = Σ(y_true - mean(y_true))²
```

**5. Adjusted R²**
```python
Adj_R² = 1 - ((1-R²) * (n-1)) / (n-p-1)
```

### Key Implementation Details:

**1. Array Operations:**
- Use NumPy for efficient computation
- Vectorize operations (avoid loops)
- Ensure arrays have same shape

**2. Edge Cases:**
- Division by zero in R² (when SS_tot = 0)
- Empty arrays
- Single value arrays

**3. Numerical Stability:**
- Handle very small/large values
- Use appropriate precision

**4. Validation:**
- Test against sklearn implementations
- Use known test cases
- Verify mathematical relationships

<a name='ex-6'></a>
### Exercise 6 - Implement All Metrics from Scratch

**Your Task:**

Implement from scratch:
1. `mse_scratch(y_true, y_pred)`
2. `rmse_scratch(y_true, y_pred)`
3. `mae_scratch(y_true, y_pred)`
4. `r2_scratch(y_true, y_pred)`
5. `adjusted_r2_scratch(y_true, y_pred, n_features)`
6. Validate against sklearn implementations

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For MSE:**
* Residuals: `y_true - y_pred`
* Squared: `residuals**2`
* Mean: `np.mean(squared)`

**For RMSE:**
* Calculate MSE first
* Take square root: `np.sqrt(mse)`

**For MAE:**
* Absolute residuals: `np.abs(residuals)`
* Mean: `np.mean(absolute)`

**For R²:**
* SS_res: `np.sum((y_true - y_pred)**2)`
* SS_tot: `np.sum((y_true - y_true.mean())**2)`
* R²: `1 - (ss_res / ss_tot)`
* Handle division by zero

**For Adjusted R²:**
* Calculate R² first
* Apply adjustment: `1 - ((1 - r2) * (n - 1)) / (n - p - 1)`

</details>

#### 6a. `mse_scratch` - Mean Squared Error

Implement **MSE from scratch** without using any library metric functions. MSE penalises larger errors more heavily due to the squaring, making it sensitive to outliers.

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [ ]:
# GRADED FUNCTION: mse_scratch
def mse_scratch(y_true, y_pred):
    """
    Calculate Mean Squared Error from scratch.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
    
    Returns:
        MSE value
    """
    ### START CODE HERE ### (≈ 5 lines)
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    mse = None 
    
    ### END CODE HERE ###
    
    return mse

In [ ]:
unittests.exercise_6a(mse_scratch)

#### 6b. `rmse_scratch` - Root Mean Squared Error

Implement **RMSE from scratch**. RMSE is the square root of MSE, which brings the error back to the **same unit** as the target variable, making it easier to interpret than MSE directly.

$$RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

In [ ]:
# GRADED FUNCTION: rmse_scratch
def rmse_scratch(y_true, y_pred):
    """
    Calculate Root Mean Squared Error from scratch.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
    
    Returns:
        RMSE value
    """
    ### START CODE HERE ### (≈ 3 lines)
    
    mse = mse_scratch(y_true, y_pred)
    rmse = None
    
    ### END CODE HERE ###
    
    return rmse

In [ ]:
unittests.exercise_6b(rmse_scratch)

#### 6c. `mae_scratch` - Mean Absolute Error

Implement **MAE from scratch**. Unlike MSE, MAE uses absolute values instead of squares, making it **more robust to outliers** — all errors contribute equally regardless of their magnitude.

$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

In [ ]:
# GRADED FUNCTION: mae_scratch
def mae_scratch(y_true, y_pred):
    """
    Calculate Mean Absolute Error from scratch.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
    
    Returns:
        MAE value
    """
    ### START CODE HERE ### (≈ 5 lines)
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    mae = None
    
    ### END CODE HERE ###
    
    return mae

In [ ]:
unittests.exercise_6c(mae_scratch)

#### 6d. `r2_scratch` - R-Squared Coefficient

Implement **R² from scratch**. R² measures the proportion of variance in the target that is explained by the model. A value of **1.0** means perfect predictions; **0.0** means the model is no better than always predicting the mean. Handle the edge case where the target is constant.

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

In [ ]:
# GRADED FUNCTION: r2_scratch
def r2_scratch(y_true, y_pred):
    """
    Calculate R-squared from scratch.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
    
    Returns:
        R² value
    """
    ### START CODE HERE ### (≈ 10 lines)
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Calculate sum of squares
    ss_res = None  
    ss_tot = None 
    
    # Handle edge case: constant target
    if ss_tot == 0:
        return 0.0 if ss_res == 0 else float('-inf')
    
    r2 = None  
    
    ### END CODE HERE ###
    
    return r2

In [ ]:
unittests.exercise_6d(r2_scratch)

#### 6e. `adjusted_r2_scratch` - Adjusted R-Squared

Implement **Adjusted R² from scratch**. Adjusted R² penalises adding features that don't meaningfully improve the model — unlike plain R², it can **decrease** when irrelevant features are added, making it more reliable for comparing models with different numbers of features.

$$\bar{R}^2 = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$$

where $n$ is the number of samples and $p$ is the number of features.

In [ ]:
# GRADED FUNCTION: adjusted_r2_scratch
def adjusted_r2_scratch(y_true, y_pred, n_features):
    """
    Calculate Adjusted R-squared from scratch.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
        n_features: Number of features
    
    Returns:
        Adjusted R² value
    """
    ### START CODE HERE ### (≈ 8 lines)
    
    r2 = r2_scratch(y_true, y_pred)
    
    n = None
    p = None
    
    # Apply adjustment formula
    adjusted_r2 = None  
    ### END CODE HERE ###
    
    return adjusted_r2

In [ ]:
unittests.exercise_6e(adjusted_r2_scratch)

In [ ]:
# Test implementations
print("="*60)
print("FROM-SCRATCH IMPLEMENTATION VALIDATION")
print("="*60)

# Test with simple example first
y_true_test = np.array([3.0, -0.5, 2.0, 7.0])
y_pred_test = np.array([2.5, 0.0, 2.0, 8.0])

print("\nSimple Test Case:")
print(f"y_true: {y_true_test}")
print(f"y_pred: {y_pred_test}")

print("\nScratch Implementations:")
mse_s = mse_scratch(y_true_test, y_pred_test)
rmse_s = rmse_scratch(y_true_test, y_pred_test)
mae_s = mae_scratch(y_true_test, y_pred_test)
r2_s = r2_scratch(y_true_test, y_pred_test)
adj_r2_s = adjusted_r2_scratch(y_true_test, y_pred_test, n_features=2)

print(f"  MSE:          {mse_s:.6f}")
print(f"  RMSE:         {rmse_s:.6f}")
print(f"  MAE:          {mae_s:.6f}")
print(f"  R²:           {r2_s:.6f}")
print(f"  Adjusted R²:  {adj_r2_s:.6f}")

# Test on actual dataset
print("\n" + "="*60)
print("CALIFORNIA HOUSING DATASET VALIDATION")
print("="*60)

metrics_scratch = {
    'MSE': mse_scratch(y_test, y_pred_test),
    'RMSE': rmse_scratch(y_test, y_pred_test),
    'MAE': mae_scratch(y_test, y_pred_test),
    'R²': r2_scratch(y_test, y_pred_test),
    'Adj R²': adjusted_r2_scratch(y_test, y_pred_test, X_test.shape[1])
}

metrics_sklearn = {
    'MSE': mean_squared_error(y_test, y_pred_test),
    'RMSE': mean_squared_error(y_test, y_pred_test, squared=False),
    'MAE': mean_absolute_error(y_test, y_pred_test),
    'R²': r2_score(y_test, y_pred_test),
    'Adj R²': 1 - ((1 - r2_score(y_test, y_pred_test)) * (len(y_test) - 1)) / (len(y_test) - X_test.shape[1] - 1)
}

print(f"\n{'Metric':<12} {'Scratch':<15} {'Sklearn':<15} {'Difference':<15} {'Match'}")
print("-"*70)
for metric_name in metrics_scratch:
    scratch_val = metrics_scratch[metric_name]
    sklearn_val = metrics_sklearn[metric_name]
    diff = abs(scratch_val - sklearn_val)
    match = '✓' if diff < 1e-10 else '✗'
    print(f"{metric_name:<12} {scratch_val:<15.10f} {sklearn_val:<15.10f} {diff:<15.2e} {match}")

# Overall validation
all_match = all(abs(metrics_scratch[m] - metrics_sklearn[m]) < 1e-10 for m in metrics_scratch)
print("\n" + "="*60)
if all_match:
    print("✓ ALL IMPLEMENTATIONS CORRECT!")
else:
    print("✗ Some implementations need fixing")
print("="*60)

# Verify mathematical relationships
print("\nMathematical Relationships:")
print(f"  RMSE = √MSE: {abs(rmse_scratch(y_test, y_pred_test) - np.sqrt(mse_scratch(y_test, y_pred_test))) < 1e-10}")
print(f"  MAE ≤ RMSE:  {mae_scratch(y_test, y_pred_test) <= rmse_scratch(y_test, y_pred_test)}")
print(f"  Adj R² ≤ R²: {adjusted_r2_scratch(y_test, y_pred_test, X_test.shape[1]) <= r2_scratch(y_test, y_pred_test)}")

##### **Expected Output**
```
============================================================
FROM-SCRATCH IMPLEMENTATION VALIDATION
============================================================

Simple Test Case:
y_true: [ 3.  -0.5  2.   7. ]
y_pred: [2.5 0.  2.  8. ]

Scratch Implementations:
  MSE:          0.375000
  RMSE:         0.612372
  MAE:          0.500000
  R²:           0.948608
  Adjusted R²:  0.897216

============================================================
CALIFORNIA HOUSING DATASET VALIDATION
============================================================

Metric       Scratch         Sklearn         Difference      Match
----------------------------------------------------------------------
MSE          0.XXXXXXXXXX    0.XXXXXXXXXX    X.XXe-XX        ✓
...

============================================================
✓ ALL IMPLEMENTATIONS CORRECT!
============================================================
```
All your from-scratch implementations should match sklearn exactly (difference < 1e-10).

<a name='8'></a>
## 8 - Conclusion

**Congratulations!** You've completed the Regression Metrics assignment!

### What You've Learned:

1. **Error Metrics**: Understanding MSE, RMSE, MAE and their differences
2. **R-Squared**: Interpreting explained variance and Adjusted R²
3. **Residual Analysis**: Diagnosing model assumptions and identifying issues
4. **Metric Comparison**: Evaluating models comprehensively across multiple metrics
5. **Custom Metrics**: Designing business-specific loss functions
6. **From-Scratch Implementation**: Deep understanding through manual calculation

### Key Takeaways:

**Metric Selection:**
- **MSE/RMSE**: When large errors are very costly
- **MAE**: When all errors matter equally, robust to outliers
- **R²**: For comparing models, understanding explained variance
- **Adjusted R²**: When comparing models with different feature counts
- **Custom metrics**: When business costs don't align with standard metrics

**Best Practices:**
- Always check residual plots (more important than metrics alone)
- Compare training vs test metrics (detect overfitting)
- Use multiple metrics for comprehensive evaluation
- Consider outliers and their impact
- Align metrics with business objectives
- Validate model assumptions

**Common Pitfalls:**

1. **Relying solely on R²**
   - Solution: Always check residual plots and multiple metrics

2. **Ignoring outliers**
   - Solution: Compare RMSE vs MAE, identify and handle outliers

3. **Not checking assumptions**
   - Solution: Perform residual analysis (normality, homoscedasticity)

4. **Using wrong metric for problem**
   - Solution: Understand business costs, use custom metrics if needed

5. **Overfitting to validation metrics**
   - Solution: Use holdout test set, cross-validation

### Real-World Applications:

**Real Estate Pricing:**
- MAPE for relative accuracy across price ranges
- Custom asymmetric loss (over-pricing vs under-pricing)
- Residual analysis to identify pricing anomalies

**Demand Forecasting:**
- Asymmetric loss (stockout vs excess inventory)
- Quantile regression for safety stock
- MAE for robust predictions

**Energy Consumption:**
- RMSE for capacity planning
- R² for model comparison
- Residual analysis for usage patterns

**Financial Modeling:**
- Multiple metrics for comprehensive evaluation
- Custom metrics for risk-adjusted returns
- Threshold accuracy for acceptable error ranges

### Advanced Topics:

**Cross-Validation:**
- K-fold cross-validation for robust evaluation
- Time series cross-validation for temporal data
- Stratified sampling for consistent distributions

**Confidence Intervals:**
- Prediction intervals for uncertainty quantification
- Bootstrap methods for metric confidence
- Bayesian approaches for probabilistic predictions

**Advanced Residual Analysis:**
- Cook's distance for influential points
- Leverage plots for high-leverage observations
- Partial residual plots for feature relationships
- Durbin-Watson test for autocorrelation

**Robust Regression:**
- Huber loss (combines MSE and MAE)
- RANSAC for outlier-resistant fitting
- Quantile regression for robust predictions

### Diagnostic Checklist:

**Before Trusting Your Model:**
- [ ] Multiple metrics calculated (MSE, MAE, R²)
- [ ] Training vs test performance compared
- [ ] Residual plot shows no patterns
- [ ] Residuals approximately normally distributed
- [ ] Constant variance across predictions (homoscedastic)
- [ ] Outliers identified and investigated
- [ ] Business-relevant metrics evaluated
- [ ] Cross-validation performed
- [ ] Model makes intuitive sense

### Tools and Libraries:

**Evaluation:**
- scikit-learn: Comprehensive metrics library
- scipy.stats: Statistical tests for residuals
- statsmodels: Advanced regression diagnostics
- Yellowbrick: Visualization for regression evaluation

**Monitoring:**
- Evidently: ML monitoring for regression
- WhyLogs: Data quality and drift detection
- Grafana: Real-time metric dashboards

**Experimentation:**
- MLflow: Track metrics across experiments
- Weights & Biases: Visualize metric evolution
- Optuna: Hyperparameter optimization with custom metrics

### Recommended Resources:

- **Papers**:
  - "A Survey of Evaluation Metrics for Machine Learning" (various)
  - "Regression Diagnostics" (Belsley et al., 2005)
  - "Quantile Regression" (Koenker, 2005)

- **Books**:
  - "Applied Linear Regression" (Weisberg, 2013)
  - "Introduction to Linear Regression Analysis" (Montgomery et al., 2021)
  - "Regression Modeling Strategies" (Harrell, 2015)

- **Documentation**:
  - scikit-learn metrics guide
  - statsmodels regression diagnostics
  - scipy.stats statistical tests

**Excellent work! You now understand how to properly evaluate regression models, diagnose issues, and create custom metrics aligned with business objectives!**